In [1]:
import torch
from transformers import pipeline
import textwrap
import pandas as pd
from tqdm import tqdm 

pipe = pipeline(
        "text-generation",
        model="meta-llama/Llama-3.2-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="auto",
    )

c:\Users\oscr1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.96s/it]


In [2]:
def generate_summary(story):
    
    messages = [
        {
            "role": "system",
            "content": textwrap.dedent(
                """\
                You are a summarizer who writes concise and informative summaries of stories.
                Each summary should capture the main points and essence of the story in one sentence.
                """
            ).replace("\n", " "),
        },
        {
            "role": "user",
            "content": textwrap.dedent(
                f"""\ 
                Summarize the following story in one sentence: {story}
                """
            ).replace("\n", " "),
        },
    ]

    outputs = pipe(
        messages,
        max_new_tokens=50, 
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    return outputs[0]["generated_text"][-1]["content"]



In [3]:
df = pd.read_csv('train.csv')

In [18]:
df = df.head(20000)

In [27]:
len(df['text'])

20000

In [28]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [30]:
def process_example(example):
    example['summary'] = generate_summary(example['text'])
    return example

In [31]:
dataset = dataset.map(process_example, batched=False, desc="Generating summaries")

Generating summaries: 100%|██████████| 20000/20000 [5:41:06<00:00,  1.02s/ examples]  


In [32]:
summary_df = dataset.to_pandas()

In [33]:
summary_df.to_csv('train_with_summaries.csv', index=False)

In [37]:
summary_df.head()

,text,summary
0,"One day, a little girl named Lily found a need...","Lily, a little girl, and her mother successful..."
1,"Once upon a time, there was a little car named...","A happy little car named Beep, fueled by good ..."
2,"One day, a little fish named Fin was swimming ...",A little fish named Fin befriends a crab who i...
3,"Once upon a time, in a land full of trees, the...","A small, weak cherry tree finds happiness and ..."
4,"Once upon a time, there was a little girl name...","A young girl named Lily, along with her cat an..."


In [38]:
len(summary_df)

20000